# Libraries import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib

sns.set_theme(style="whitegrid")

# Load Data & Recreate Part 2 Preprocessing

In [ ]:
df = pd.read_csv('data/cleaned_data.csv')
df['Traffic Control Presence'] = df['Traffic Control Presence'].fillna('None')
df['Driver License Status'] = df['Driver License Status'].fillna('Unknown')

y_clf = (df['Accident Severity'] == 'Fatal').astype(int)
X = df.drop(columns=['Number of Casualties', 'Accident Severity'])

def get_time_bucket(time_str):
    hour = int(time_str.split(':')[0])
    if 5 <= hour < 12: return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening'
    else: return 'Night'

X['Time Bucket'] = X['Time of Day'].apply(get_time_bucket)
X = X.drop(columns=['City Name', 'Time of Day'])

categorical_cols = X.select_dtypes(include='object').columns.tolist()
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

X_train, X_test, y_clf_train, y_clf_test = train_test_split(
    X_encoded, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train.shape, X_test.shape

## Decision Tree — Unconstrained Baseline

In [ ]:
dt_unconstrained = DecisionTreeClassifier(max_depth=None, random_state=42)
dt_unconstrained.fit(X_train_scaled, y_clf_train)

train_acc = accuracy_score(y_clf_train, dt_unconstrained.predict(X_train_scaled))
test_acc = accuracy_score(y_clf_test, dt_unconstrained.predict(X_test_scaled))

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

## Decision Tree — Controlled (max_depth=5, min_samples_split=20)

In [ ]:
dt_controlled = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
dt_controlled.fit(X_train_scaled, y_clf_train)

train_acc_c = accuracy_score(y_clf_train, dt_controlled.predict(X_train_scaled))
test_acc_c = accuracy_score(y_clf_test, dt_controlled.predict(X_test_scaled))

print(f"Training Accuracy: {train_acc_c:.4f}")
print(f"Test Accuracy: {test_acc_c:.4f}")
print(f"\nTrain-Test Gap (Unconstrained): {train_acc - test_acc:.4f}")
print(f"Train-Test Gap (Controlled): {train_acc_c - test_acc_c:.4f}")

## Gini vs Entropy Comparison

In [ ]:
dt_gini = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
dt_gini.fit(X_train_scaled, y_clf_train)
acc_gini = accuracy_score(y_clf_test, dt_gini.predict(X_test_scaled))

dt_entropy = DecisionTreeClassifier(max_depth=5, criterion='entropy', random_state=42)
dt_entropy.fit(X_train_scaled, y_clf_train)
acc_entropy = accuracy_score(y_clf_test, dt_entropy.predict(X_test_scaled))

print(f"Gini Test Accuracy: {acc_gini:.4f}")
print(f"Entropy Test Accuracy: {acc_entropy:.4f}")

## Random Forest

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train_scaled, y_clf_train)

rf_train_acc = accuracy_score(y_clf_train, rf_model.predict(X_train_scaled))
rf_test_acc = accuracy_score(y_clf_test, rf_model.predict(X_test_scaled))
rf_test_auc = roc_auc_score(y_clf_test, rf_model.predict_proba(X_test_scaled)[:, 1])

print(f"Training Accuracy: {rf_train_acc:.4f}")
print(f"Test Accuracy: {rf_test_acc:.4f}")
print(f"Test ROC-AUC: {rf_test_auc:.4f}")

## Random Forest — Feature Importance

In [ ]:
rf_importances = pd.Series(rf_model.feature_importances_, index=X_encoded.columns)
top5_rf = rf_importances.sort_values(ascending=False).head(5)
print(top5_rf)

plt.figure(figsize=(8, 5))
sns.barplot(x=top5_rf.values, y=top5_rf.index, hue=top5_rf.index, palette='rocket', legend=False)
plt.title('Top 5 Feature Importances — Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('plots/rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Gradient Boosting

In [ ]:
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_model.fit(X_train_scaled, y_clf_train)

gb_train_acc = accuracy_score(y_clf_train, gb_model.predict(X_train_scaled))
gb_test_acc = accuracy_score(y_clf_test, gb_model.predict(X_test_scaled))
gb_test_auc = roc_auc_score(y_clf_test, gb_model.predict_proba(X_test_scaled)[:, 1])

print(f"Training Accuracy: {gb_train_acc:.4f}")
print(f"Test Accuracy: {gb_test_acc:.4f}")
print(f"Test ROC-AUC: {gb_test_auc:.4f}")

## Feature Ablation Study — Removing Least Important Features

In [ ]:
lowest5_features = rf_importances.sort_values(ascending=True).head(5).index.tolist()
print("5 lowest-importance features to remove:")
print(lowest5_features)

X_train_reduced = pd.DataFrame(X_train_scaled, columns=X_encoded.columns).drop(columns=lowest5_features)
X_test_reduced = pd.DataFrame(X_test_scaled, columns=X_encoded.columns).drop(columns=lowest5_features)

rf_reduced = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_reduced.fit(X_train_reduced, y_clf_train)

auc_full = roc_auc_score(y_clf_test, rf_model.predict_proba(X_test_scaled)[:, 1])
auc_reduced = roc_auc_score(y_clf_test, rf_reduced.predict_proba(X_test_reduced)[:, 1])

print(f"\nFull model AUC (all features): {auc_full:.4f}")
print(f"Reduced model AUC (5 lowest-importance features removed): {auc_reduced:.4f}")

## Cross-Validated Comparison — 5-Fold AUC

In [ ]:
models_to_compare = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Decision Tree (max_depth=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, model in models_to_compare.items():
    scores = cross_val_score(model, X_train_scaled, y_clf_train, cv=cv, scoring='roc_auc')
    cv_results.append({'Model': name, 'Mean AUC': scores.mean(), 'Std AUC': scores.std()})
    print(f"{name}: Mean AUC = {scores.mean():.4f}, Std = {scores.std():.4f}")

cv_results_df = pd.DataFrame(cv_results)

## Hyperparameter Tuning — GridSearchCV

In [ ]:
pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}

grid_search = GridSearchCV(
    pipeline, param_grid, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc', n_jobs=-1
)

grid_search.fit(X_train, y_clf_train)

print("Best Params:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

## Manual Learning Curve

In [ ]:
best_pipeline = grid_search.best_estimator_

fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
learning_curve_results = []

for f in fractions:
    n_rows = int(f * len(X_train))
    X_subset = X_train.iloc[:n_rows]
    y_subset = y_clf_train.iloc[:n_rows]
    
    best_pipeline.fit(X_subset, y_subset)
    
    train_proba = best_pipeline.predict_proba(X_subset)[:, 1]
    train_auc = roc_auc_score(y_subset, train_proba)
    
    test_proba = best_pipeline.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_clf_test, test_proba)
    
    learning_curve_results.append({'Training Fraction': f, 'Training AUC': train_auc, 'Test AUC': test_auc})
    print(f"Fraction {f}: Train AUC = {train_auc:.4f}, Test AUC = {test_auc:.4f}")

learning_curve_df = pd.DataFrame(learning_curve_results)

## Serialize Best Model

In [ ]:
best_pipeline.fit(X_train, y_clf_train)
joblib.dump(best_pipeline, 'best_model.pkl')
print("Model saved as best_model.pkl")

## Reload and Predict — Verification

In [ ]:
loaded_model = joblib.load('best_model.pkl')

sample_1 = X_test.iloc[[0]]
sample_2 = X_test.iloc[[1]]

pred_1 = loaded_model.predict(sample_1)
pred_2 = loaded_model.predict(sample_2)

print(f"Sample 1 prediction: {pred_1[0]} (actual: {y_clf_test.iloc[0]})")
print(f"Sample 2 prediction: {pred_2[0]} (actual: {y_clf_test.iloc[1]})")
print("\nReload and predict successful — no errors.")

## Summary Comparison — All Models (Parts 2 & 3)

In [ ]:
summary_table = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree (max_depth=5)', 
              'Random Forest', 'Gradient Boosting', 'Random Forest (GridSearch-tuned)'],
    '5-Fold CV Mean AUC': [0.5207, 0.5222, 0.5197, 0.5290, 0.5252],
    '5-Fold CV Std AUC': [0.0187, 0.0361, 0.0247, 0.0297, None],
    'Test-Set AUC': [0.5232, None, 0.5064, 0.4950, None]
})
print(summary_table)